In [1]:
# Notebook setup: import and (optionally) auto-reload local module
import importlib
import time

import locator_time
import locator_time2

# If you edit locator_time2.py while this notebook is open, re-run this cell to reload.
importlib.reload(locator_time2)

print(locator_time2.__doc__[:400])

A small, self-contained Python port of ImPlot's time-axis tick locator.

This is based on the logic in `implot.cpp` (function `Locator_Time`) and the
associated helpers in the "Time Ticks and Utils" section.

Goal
----
Given:
  - `t_min` and `t_max` as Unix timestamps in seconds (float or int)
  - `pixels` as the axis pixel length (float or int)

Produce:
  - tick positions (float seconds)
  - lab


## Test out the Data Structures

In [2]:
# test out ImPlotTime from locator_time2
locator_time2.ImPlotTime(1, 1)

ImPlotTime(S=1, Us=1)

In [3]:
# arithmetic + normalization
(locator_time2.ImPlotTime(3, 1) + locator_time2.ImPlotTime(1, 2_000_000)).to_double()

6.000001

## Test out the helper functions

In [4]:
# Small helpers to inspect returned ticks
from collections import Counter

def summarize_ticks(ticks):
    c = Counter((t.level, t.major, t.show_label) for t in ticks)
    total = len(ticks)
    level0 = sum(1 for t in ticks if t.level == 0)
    level1 = sum(1 for t in ticks if t.level == 1)
    shown = sum(1 for t in ticks if t.show_label)
    return {
        'total_ticks': total,
        'level0_ticks': level0,
        'level1_ticks': level1,
        'labels_shown': shown,
        'breakdown': dict(c),
    }

def head_ticks(ticks, n=20):
    rows = []
    for t in ticks[:n]:
        tag = f"L{t.level} {'M' if t.major else 'm'}"
        label = t.label if t.show_label else ''
        rows.append((tag, t.pos, label))
    return rows

## Baseline: 1 hour range
This should produce minute/second-ish ticks depending on `pixels` and `max_density`.

In [5]:
now = time.time()
t_min = now
t_max = now + 3600

ticks = locator_time2.locator_time(
    t_min, t_max, pixels=800,
    use_local_time=True,
    max_density=0.5,
    char_px=7.0,
)

summarize_ticks(ticks), head_ticks(ticks, 25)

({'total_ticks': 7,
  'level0_ticks': 6,
  'level1_ticks': 1,
  'labels_shown': 7,
  'breakdown': {(0, True, True): 1, (1, True, True): 1, (0, False, True): 5}},
 [('L0 M', 1766977200.0, '10:00pm'),
  ('L1 M', 1766977200.0, '12/28/25'),
  ('L0 m', 1766977800.0, '10:10pm'),
  ('L0 m', 1766978400.0, '10:20pm'),
  ('L0 m', 1766979000.0, '10:30pm'),
  ('L0 m', 1766979600.0, '10:40pm'),
  ('L0 m', 1766980200.0, '10:50pm')])

## Compare pixel widths
Smaller `pixels` should suppress more labels (especially level 0 minor labels).

In [ ]:
for px in (200, 400, 800, 1200):
    ticks_px = locator_time2.locator_time(t_min, t_max, pixels=px, use_local_time=True)
    s = summarize_ticks(ticks_px)
    print(f"pixels={px:4d}  total={s['total_ticks']:4d}  shown={s['labels_shown']:4d}  L0={s['level0_ticks']:4d}  L1={s['level1_ticks']:4d}")

## Time the locator_time function

In [7]:
import timeit

In [20]:
# use timeit magic to benchmark locator_time2.locator_time
timeit.timeit(
    "locator_time2.locator_time(t_min, t_max, pixels=800, use_local_time=True)",
    globals=globals(),
    number=1000,
)

0.12127589993178844

## Explore different spans
These cover typical unit transitions (minutes → hours → days → months → years).

In [21]:
def run_span(span_seconds, pixels=900, title=None):
    t0 = time.time()
    t1 = t0 + span_seconds
    ticks = locator_time2.locator_time(t0, t1, pixels=pixels, use_local_time=True)
    s = summarize_ticks(ticks)
    title = title or f"span={span_seconds}s"
    print(f"\n{title} (pixels={pixels})")
    print(f"  total={s['total_ticks']}  shown={s['labels_shown']}")
    print('  first 12:', head_ticks(ticks, 12))

run_span(10, title='10 seconds')
run_span(5 * 60, title='5 minutes')
run_span(6 * 3600, title='6 hours')
run_span(2 * 86400, title='2 days')
run_span(45 * 86400, title='45 days')
run_span(400 * 86400, title='~400 days (year-ish)')
run_span(10 * 365 * 86400, title='~10 years (year locator)')


10 seconds (pixels=900)
  total=11  shown=11
  first 12: [('L0 m', 1767020118.0, ':18'), ('L1 M', 1767020118.0, '12/29/25 9:55am'), ('L0 m', 1767020119.0, ':19'), ('L0 m', 1767020120.0, ':20'), ('L0 m', 1767020121.0, ':21'), ('L0 m', 1767020122.0, ':22'), ('L0 m', 1767020123.0, ':23'), ('L0 m', 1767020124.0, ':24'), ('L0 m', 1767020125.0, ':25'), ('L0 m', 1767020126.0, ':26'), ('L0 m', 1767020127.0, ':27')]

5 minutes (pixels=900)
  total=26  shown=26
  first 12: [('L0 m', 1767020130.0, ':30'), ('L1 M', 1767020130.0, '12/29/25 9:55am'), ('L0 m', 1767020145.0, ':45'), ('L0 M', 1767020160.0, ':00'), ('L1 M', 1767020160.0, '9:56am'), ('L0 m', 1767020175.0, ':15'), ('L0 m', 1767020190.0, ':30'), ('L0 m', 1767020205.0, ':45'), ('L0 M', 1767020220.0, ':00'), ('L1 M', 1767020220.0, '9:57am'), ('L0 m', 1767020235.0, ':15'), ('L0 m', 1767020250.0, ':30')]

6 hours (pixels=900)
  total=12  shown=7
  first 12: [('L0 M', 1767020400.0, '10:00am'), ('L1 M', 1767020400.0, '12/29/25'), ('L0 M', 17670

## ISO-8601 / 24-hour formatting toggles
These flags match the knobs you might want in a UI layer.

In [ ]:
t_min = time.time()
t_max = t_min + 3 * 3600

ticks_default = locator_time2.locator_time(t_min, t_max, 800, use_local_time=True, use_24_hour=False, use_iso8601=False)
ticks_iso24 = locator_time2.locator_time(t_min, t_max, 800, use_local_time=True, use_24_hour=True, use_iso8601=True)

print('default:', head_ticks(ticks_default, 10))
print('iso+24:', head_ticks(ticks_iso24, 10))

## TimeAxisLocator wrapper
This exercises the reusable class intended for fast zoom callbacks.

In [ ]:
loc = locator_time2.TimeAxisLocator(use_local_time=True, prewarm=True)
ticks2 = loc(time.time(), time.time() + 3600, 800)
summarize_ticks(ticks2), head_ticks(ticks2, 25)

## Optional: PIL-based text measurement
If you want more ImPlot-like behavior, measure string widths using the same font file + size your DearCyGui axis labels use.

This requires Pillow (`pip install pillow`). If Pillow/font loading fails, it will fall back to the `char_px` estimator.

In [ ]:
# TODO: set these to match your DearCyGui axis font
font_path = None  # e.g. r"C:\\path\\to\\yourfont.otf"
font_size_px = None  # e.g. 17

measure = None
if font_path is not None and font_size_px is not None:
    try:
        measure = locator_time2.make_pil_text_width_measurer(font_path, font_size_px)
        print('PIL measurer enabled')
    except Exception as e:
        print('PIL measurer not available, falling back:', e)
        measure = None

loc_pil = locator_time2.TimeAxisLocator(
    use_local_time=True,
    measure_text_width_px=measure,
    prewarm=True,
)
ticks_pil = loc_pil(time.time(), time.time() + 3600, 800)
summarize_ticks(ticks_pil), head_ticks(ticks_pil, 25)